# CZKB — Baseline Judge Run (local / Databricks dual-mode)

JIRA: **GCAI-3092** — diagnostic LLM-as-a-Judge evaluation for the CZ Knowledge-Base chatbot.

This notebook runs the `hg_ds_evals` judge against the CZKB input table.

- **YAML config:** [`configs/czkb_exp_001_baseline_no_expected_enums.yaml`](../configs/czkb_exp_001_baseline_no_expected_enums.yaml)
- **System template:** [`configs/system.md.j2`](../configs/system.md.j2)
- **Judge:** `gpt-5-1` via `databricks_async`, `reasoning_effort: high`
- **Output:** CSV checkpoint under the directory set by `paths.checkpoint_dir` in the YAML (currently `checkpoints/`, relative to the notebook working directory).

## Run modes
- `RUN_ON_DBX = True` — runs on a Databricks cluster, reads the UC table via `spark.read.table(...)`.
- `RUN_ON_DBX = False` — runs on your laptop, reads the UC table via `databricks-sql-connector` against a SQL warehouse, authenticates with the named CLI profile, writes results to the local `checkpoints/` directory.

> **NOTE — why we do not call `run_experiment` directly:** the upstream
> `run_experiment` helper instantiates `PromptBuilder(rubric=rubric)`
> without passing the YAML's `paths.system_template_path`, so the
> default embedded template is used and our `critical_evaluation_rules`,
> `domain_specific_guidance`, `final_reminders`, and the baked-in
> **AGENT CONTEXT** are silently dropped. To keep the custom template
> in effect we replicate `run_experiment`'s flow explicitly below.

Run 
`databricks auth login --host https://adb-3174992876438447.7.azuredatabricks.net` to prevent the auth errors.

## Run mode

In [1]:
# Toggle to run on a Databricks cluster vs. locally from your laptop.
RUN_ON_DBX: bool = False

# Local-only: which Databricks CLI profile to use for auth.
DBX_PROFILE: str = "adb-chat"

DBX_CLUSTER_ID: str | None = "0424-122149-m97focly"
DBX_SQL_WAREHOUSE_ID: str | None = None

In [2]:
!databricks auth describe --profile adb-chat   # OK
!databricks current-user me  --profile adb-chat # OK

]11;?\Host: https://adb-3174992876438447.7.azuredatabricks.net
User: sg7cb@s-mxs.net
Authenticated with: metadata-service
-----
Current configuration:
  ✓ host: https://adb-3174992876438447.7.azuredatabricks.net (from DATABRICKS_HOST environment variable)
  ✓ cluster_id: 0424-122149-m97focly (from DATABRICKS_CLUSTER_ID environment variable)
  ✓ metadata_service_url: ******** (from DATABRICKS_METADATA_SERVICE_URL environment variable)
  ✓ account_id: 85cde0e4-68ed-4a99-b884-abaec50d1f90 (from /Users/SG7CB/.databrickscfg config file)
  ✓ workspace_id: 3174992876438447 (from /Users/SG7CB/.databrickscfg config file)
  ✓ profile: adb-chat (from --profile flag)
  ✓ auth_type: metadata-service (from DATABRICKS_AUTH_TYPE environment variable)
  ✓ cloud: Azure
  ✓ audience: https://adb-3174992876438447.7.azuredatabricks.net/oidc/v1/token
  ✓ discovery_url: https://adb-3174992876438447.7.azuredatabricks.net/oidc/.well-known/oauth-authorization-server
]11;?\{
  "active":true,
  "displayName":

## Environment & imports

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
if RUN_ON_DBX:
    # Cluster-only bootstrap. Skipped locally.
    !pip install -qe /Workspace/Users/sg7cb@s-mxs.net/hey-george-ds/ds_common
    !pip install openai -q
    dbutils.library.restartPython()  # noqa: F821 (provided by DBR)
else:
    print("Local mode — skipping cluster pip installs / restartPython.")
    print("Install once into your venv if missing:")
    print("    uv pip install --system-certs databricks-sql-connector openai")

Local mode — skipping cluster pip installs / restartPython.
Install once into your venv if missing:
    uv pip install --system-certs databricks-sql-connector openai


In [5]:
import os
import sys
from pathlib import Path

# Make the local hg-ds-evals repo importable when running off-cluster.
if not RUN_ON_DBX:
    _repo_root = Path.cwd()
    while _repo_root != _repo_root.parent and not (_repo_root / "hg_ds_evals").is_dir():
        _repo_root = _repo_root.parent
    if (_repo_root / "hg_ds_evals").is_dir() and str(_repo_root) not in sys.path:
        sys.path.insert(0, str(_repo_root))
        print(f'Local repo root added to sys.path: "{_repo_root}"')

    os.environ["DATABRICKS_CONFIG_PROFILE"] = DBX_PROFILE

    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    print(f"Authenticated as: {_w.current_user.me().user_name}")
    print(f"Workspace host:   {_w.config.host}")

import config_nbs
config_nbs.add_local_paths()

import hg_ds_evals
print(hg_ds_evals.__file__)

import pandas as pd
pd.set_option("display.width", None)

Authenticated as: sg7cb@s-mxs.net
Workspace host:   https://adb-3174992876438447.7.azuredatabricks.net
Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/" added!
Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/czkb/" added!
/Users/SG7CB/Developer/hg-ds-evals/hg_ds_evals/__init__.py


In [6]:
from databricks.sdk import WorkspaceClient
for w in WorkspaceClient().warehouses.list():
    print(w.id, w.state, w.name)

eab78c481da6edd8 State.RUNNING Serverless Starter Warehouse


## UC table reader

Local: `databricks-sql-connector` against a SQL warehouse, authenticating via the same CLI profile.
Cluster: `spark.read.table(...)`.

In [7]:
def _resolve_http_path(workspace_client, *, cluster_id=None, warehouse_id=None) -> tuple[str, str]:
    host = workspace_client.config.host.replace("https://", "").rstrip("/")
    if cluster_id:
        # All-purpose cluster path: /sql/protocolv1/o/<org_id>/<cluster_id>
        org_id = host.split(".")[0].removeprefix("adb-").split(".")[0]
        # Better: read it from the workspace URL: adb-<org_id>.<n>.azuredatabricks.net
        org_id = host.split("adb-")[1].split(".")[0]
        return host, f"/sql/protocolv1/o/{org_id}/{cluster_id}"
    # Warehouse fallback (existing behavior)
    warehouses = list(workspace_client.warehouses.list())
    match = next((w for w in warehouses if w.id == warehouse_id), None) if warehouse_id else None
    if match is None:
        running = [w for w in warehouses if str(w.state) == "RUNNING"]
        match = (running or warehouses)[0]
    return host, f"/sql/1.0/warehouses/{match.id}"


def read_uc_table(table_fqn: str) -> pd.DataFrame:
    """Read a Unity Catalog table into pandas — works locally (SQL warehouse) and on Databricks (Spark)."""
    if RUN_ON_DBX:
        return spark.read.table(table_fqn).toPandas()  # noqa: F821 (provided by DBR)

    from databricks import sql as dbx_sql
    server_hostname, http_path = _resolve_http_path(
        _w, cluster_id=DBX_CLUSTER_ID, warehouse_id=DBX_SQL_WAREHOUSE_ID
        )
    # `access_token` callable refreshes OAuth U2M tokens transparently.
    access_token = _w.config.authenticate()["Authorization"].removeprefix("Bearer ")
    with dbx_sql.connect(
        server_hostname=server_hostname,
        http_path=http_path,
        access_token=access_token,
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_fqn}")
            return cur.fetchall_arrow().to_pandas()

## Load the experiment YAML and build the rubric

In [8]:
YAML_FILE_NAME = "czkb_exp_002_baseline_no_expected_enums"
# Custom suffix appended to the checkpoint filename so a new run does not
# overwrite a prior checkpoint. Lands as `..._<reasoning>_<CHECKPOINT_SUFFIX>[_test].csv`.
# Set to "" to keep the old behaviour. Example: "2026_04_30_score".
CHECKPOINT_SUFFIX = "online_adhoc_quiet_hawk_score"

# Databricks CLI profile (~/.databrickscfg) used both for the model-serving
# call and for unattended OAuth token refresh during long runs. Empty string
# defers to the default profile or DATABRICKS_CONFIG_PROFILE env var.
# IMPORTANT: run `databricks auth login --profile <name>` once before the
# run to seed the OAuth refresh token; subsequent token rotation is then
# automatic and does not require browser interaction.
DATABRICKS_PROFILE = "adb-chat"

# When True, drop existing `error=True` rows from the checkpoint file
# before the run starts (with timestamped backup). Lets a re-run cleanly
# retry test cases that failed for transient reasons (e.g. token expiry)
# without accumulating duplicate error rows over retry cycles.
PURGE_ERRORS_BEFORE_RUN = True
config_path = f"../configs/{YAML_FILE_NAME}.yaml"
assert Path(config_path).exists(), f"config file not found: {config_path}"

In [9]:
from hg_ds_evals.rubrics.loader import (
    load_experiment_config,
    build_rubric_from_config,
    get_experiment_name,
)

config = load_experiment_config(config_path)
rubric = build_rubric_from_config(config)

print(f"Experiment       : {config['experiment']['name']}")
print(f"Sample size      : {config['dataset']['test_num_rows']}")
print(f"Rubric name      : {rubric.metadata.name}")
print(f"Rubric persona   : {rubric.metadata.persona}")
print(f"Dimensions       : {rubric.dimension_ids}")
print(f"Input fields     : {rubric.input_field_names}")
print(f"Output fields    : {[f.name for f in rubric.output_schema.fields]}")
print(f"Pass threshold   : {rubric.pass_threshold}")
print(f"Judge model      : {config['model']['model_deployment_name']} "
      f"({config['model']['api_provider']}, reasoning={config['model']['reasoning_effort']})")
print(f"Input table      : {config['dataset']['input_schema']}.{config['dataset']['input_dataset']}")
print(f"test_num_rows    : {config['dataset']['test_num_rows']}")

Experiment       : czkb_exp_002_baseline_no_expected_enums
Sample size      : None
Rubric name      : Custom Rubric
Rubric persona   : You are an expert evaluator assessing the end-to-end quality of a Czech retail-banking knowledge-base (KB) chatbot ('Hey George'), with the goal of pinpointing concrete improvements for the bot/agent team, the retrieval/reranker team, and the KB content team.

Dimensions       : ['query_clarity', 'selection_semantic_relevance', 'selected_context_sufficiency', 'optimal_retrieved_context_adequacy', 'answer_expected_alignment', 'answer_groundedness', 'language_compliance']
Input fields     : ['user_query', 'pre_prune_enum_ids', 'post_prune_candidates_text', 'post_prune_enum_ids', 'reranked_enum_ids', 'reranked_kb_context', 'expected_response', 'agent_response']
Output fields    : ['case_scope', 'expected_reference_looks_wrong', 'expected_reference_issue_description', 'optimal_enum_selection', 'expected_answer_summary_with_optimal_context', 'extra_or_distra

## Load the input data

In [ ]:
# input_table = (
#     f"{config['dataset']['input_catalog']}."
#     f"{config['dataset']['input_schema']}."
#     f"{config['dataset']['input_dataset']}"
# )
# print(input_table)

# df_input = read_uc_table(input_table)
# print(f"rows: {len(df_input):,}")
# df_input.head()

## Build the judge's system + user prompts from the **custom** template

We explicitly pass `system_template_path` and `user_template_path` — without this the default embedded template is used and the AGENT CONTEXT / critical_evaluation_rules / final_reminders sections are silently dropped.

In [10]:
from hg_ds_evals.prompts.builder import PromptBuilder
from hg_ds_evals.prompts.common import display_prompt

system_template_path = Path(config["paths"]["system_template_path"])
user_template_path = Path(config["paths"]["user_template_path"])

if not system_template_path.is_absolute():
    system_template_path = (Path.cwd() / system_template_path).resolve()
if not user_template_path.is_absolute():
    user_template_path = (Path.cwd() / user_template_path).resolve()

assert system_template_path.exists(), f"system template missing: {system_template_path}"
assert user_template_path.exists(), f"user template missing: {user_template_path}"
print(f"system template : {system_template_path}")
print(f"user template   : {user_template_path}")

builder = PromptBuilder(
    rubric=rubric,
    system_template_path=system_template_path,
    user_template_path=user_template_path,
)
system_prompt = builder.build_system_prompt()
print(f"system prompt   : {len(system_prompt)} chars")

system template : /Users/SG7CB/Developer/hg-ds-evals/experiments/czkb/configs/system.md.j2
user template   : /Users/SG7CB/Developer/hg-ds-evals/experiments/czkb/configs/user.md.j2
system prompt   : 25385 chars


In [11]:
display_prompt(system_prompt, title="System Prompt (CZKB baseline — with AGENT CONTEXT)", font_size=12)

## Preview the user prompt on one real row

In [ ]:
sample_row = df_input.head(1).iloc[0].to_dict()

user_prompt = builder.build_user_prompt(sample_row)
display_prompt(user_prompt, title=f"User Prompt — {sample_row.get('test_case_id')}", font_size=12)

## Run the evaluation

The eval results land in the **local** `checkpoints/` directory configured by the YAML (`paths.checkpoint_dir`). On a cluster this is the workspace-mounted notebook dir; locally it's relative to your CWD.

In [11]:
from hg_ds_evals.common.utils import (
    load_checkpoint,
    filter_df_with_checkpoints,
    prepare_eval_sample,
)
from hg_ds_evals.llm.api_client import get_api_client
from hg_ds_evals.evals.evaluator import async_run_evals

experiment_name = get_experiment_name(config_path)
INPUT_CATALOG = config["dataset"]["input_catalog"]
INPUT_SCHEMA = config["dataset"]["input_schema"]
INPUT_TABLE = config["dataset"]["input_dataset"]
ID_COLUMNS = config["dataset"].get("id_columns", [])
NUM_ROWS = config["dataset"]["test_num_rows"]
CHECKPOINT_DIR = config["paths"].get("checkpoint_dir", "checkpoints")

print(f"[1/5] loading input table {INPUT_CATALOG}.{INPUT_SCHEMA}.{INPUT_TABLE}")
df = read_uc_table(f"{INPUT_CATALOG}.{INPUT_SCHEMA}.{INPUT_TABLE}")
print(f"      rows: {len(df):,}")

print("\n[2/5] preparing eval sample")
df_sample, file_name_eval = prepare_eval_sample(
    df=df,
    evals_name=experiment_name,
    reasoning_effort=config["model"]["reasoning_effort"],
    suffix=CHECKPOINT_SUFFIX or None,
    num_rows=NUM_ROWS,
)
cols = config["dataset"]["eval_columns"]
if ID_COLUMNS:
    cols = list(dict.fromkeys(ID_COLUMNS + cols))
passthrough = config["dataset"].get("passthrough_columns", [])
if passthrough:
    cols = list(dict.fromkeys(cols + passthrough))
# Tolerate columns that are missing locally (e.g. derived columns produced only by Spark notebooks).
missing_cols = [c for c in cols if c not in df_sample.columns]
if missing_cols:
    print(f"      WARN — missing columns dropped: {missing_cols}")
    cols = [c for c in cols if c in df_sample.columns]
df_eval = df_sample[cols].copy()
print(f"      eval df: {len(df_eval)} rows, {len(cols)} cols  checkpoint_file={file_name_eval}")

print("\n[3/5] loading checkpoint")
ckp_df, ckp_path = load_checkpoint(
    checkpoint_file_name=file_name_eval,
    checkpoint_dir=CHECKPOINT_DIR,
    purge_error_rows=PURGE_ERRORS_BEFORE_RUN,
)
df_eval = filter_df_with_checkpoints(df_eval, ckp_df, id_cols=ID_COLUMNS)
print(f"      {len(df_eval)} rows remain after filtering {len(ckp_df)} already-scored rows")
print(f"      checkpoint path: {ckp_path}")

print("\n[4/5] setting up API client")
client = get_api_client(
    model_deployment_name=config["model"]["model_deployment_name"],
    api_provider=config["model"]["api_provider"],
    databricks_endpoint_url=config["model"].get("databricks_endpoint_url"),
    databricks_base_url=config["model"].get("databricks_base_url"),
    databricks_workspace_host=config["model"].get("databricks_workspace_host"),
    databricks_profile=DATABRICKS_PROFILE or None,
)

print("\n[5/5] running evaluations")
results_df, metrics = await async_run_evals(
    df=df_eval,
    system_prompt=system_prompt,
    client=client,
    config=config,
    checkpoint_path=ckp_path,
    user_prompt_builder=builder.build_user_prompt,
)

print("\ndone.")
print(f"checkpoint  : {ckp_path}")
print(f"metrics     : {metrics}")

[1/5] loading input table prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_czkb_exp_002_online_adhoc_quiet_hawk_score
      rows: 1,123

[2/5] preparing eval sample
Sample size: 1_123
File name: evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv
      eval df: 1123 rows, 51 cols  checkpoint_file=evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv

[3/5] loading checkpoint
ℹ No checkpoint found, starting fresh: checkpoints/evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv
      1123 rows remain after filtering 0 already-scored rows
      checkpoint path: checkpoints/evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv

[4/5] setting up API client

[5/5] running evaluations
ℹ️ Processing batch 1: 5 tasks
🔄 Databricks credentials refreshed; retrying request
✅ Batch 1 complete: 209,045 tokens in 878.9s | Last 60s: 209,045 tokens
ℹ️ Processing batch

## Check the checkpoint

In [20]:
df_results = pd.read_csv(ckp_path)
print(f"checkpoint rows: {len(df_results)}")
print()
print("error counts:")
print(df_results.get("error", pd.Series([None]*len(df_results))).isna().value_counts(dropna=False))

dim_score_cols = [c for c in df_results.columns if c.endswith("_score")]
print()
print("dimension score summary:")
print(df_results[dim_score_cols].describe().round(2))

checkpoint rows: 10

error counts:
error
False    10
Name: count, dtype: int64

dimension score summary:
       expert_score  enum_relevance_score  query_clarity_score  \
count         10.00                 10.00                10.00   
mean           7.20                  0.66                 1.90   
std            1.55                  0.41                 0.32   
min            6.00                  0.00                 1.00   
25%            6.00                  0.46                 2.00   
50%            6.00                  0.80                 2.00   
75%            9.00                  1.00                 2.00   
max            9.00                  1.00                 2.00   

       selection_semantic_relevance_score  selected_context_sufficiency_score  \
count                               10.00                               10.00   
mean                                 1.60                                1.30   
std                                  0.70                

## Next step

Open [`czkb_001_results_viewer.ipynb`](czkb_001_results_viewer.ipynb) to aggregate scores, compute the judge-vs-expert Spearman, split missed ENUMs into retriever-vs-reranker failures, and render the HTML report.